# Setup

In [ ]:
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# Set to True if running on local machine with Google Drive Desktop mounted
# Set to False if running in Google Colab cloud
RUNNING_LOCALLY = False

# Local-only smoke-test mode. When True (and RUNNING_LOCALLY=True), reads the
# test twit/author dicts directly from `<repo>/data_sets/` and writes outputs
# to subfolders under it. Lets the test branch of this notebook run end-to-end
# without Google Drive Desktop.
USE_LOCAL_TEST_DATA = False

if RUNNING_LOCALLY:
    # --- REPO ROOT ON sys.path (so `from src.*` works locally) ---
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    if USE_LOCAL_TEST_DATA:
        BASE_PATH = Path(_REPO_ROOT) / 'data_sets'
    else:
        # Standard macOS path for Google Drive Desktop
        BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')

else:
    # Google Colab cloud path
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths used across notebooks.
# In USE_LOCAL_TEST_DATA mode, BASE_PATH already points at the test-data root,
# so `datasets_folder` is BASE_PATH itself (the test JSONs live directly in it)
# and the cleaned/network outputs go in subfolders we create on demand.
if RUNNING_LOCALLY and USE_LOCAL_TEST_DATA:
    datasets_folder  = BASE_PATH
    cleanedds_folder = BASE_PATH / 'Cleaned Data'
    networks_folder  = BASE_PATH / 'Networks'
    cleanedds_folder.mkdir(parents=True, exist_ok=True)
    networks_folder.mkdir(parents=True, exist_ok=True)
else:
    datasets_folder  = BASE_PATH / 'Data Sets'
    cleanedds_folder = BASE_PATH / 'Data Sets/Cleaned Data'
    networks_folder  = BASE_PATH / 'Data Sets/Networks/'

twits_folder        = BASE_PATH / 'Raw Data/Twits/'
test_folder         = BASE_PATH / 'Raw Data/'
literature_folder   = BASE_PATH / 'Literature/'
topic_models_folder = BASE_PATH / 'Models/Topic Modeling/'


In [ ]:
from datetime import timedelta
import json, heapq, itertools, csv
import time
import datetime
from datetime import datetime, timezone
import os
import tqdm
import pickle
import numpy as np
import pandas as pd
import random
import networkx as nx
from matplotlib import pyplot as plt
import seaborn as sns
import re, string
from collections import OrderedDict
# Command was commented out: # !pip install langdetect
# from langdetect import detect_langs
# langdetect.DetectorFactory.seed = 0

from pathlib import Path
# from google.colab import drive
# drive.mount('/content/drive')

# %%
# ──────────────────────────────────────────────────────────────────────────────
# 1. Drive Mount & Paths
# ──────────────────────────────────────────────────────────────────────────────
# drive.mount('/content/drive')

# Base project folder (Ignacio standard)
# BASE = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')
# BASE = Path('/content/drive/My Drive/AI Public Trust')

# twits_folder = BASE / 'Raw Data/Twits/'
# test_folder = BASE / 'Raw Data/'
# print("Current Directory:", twits_folder)
# datasets_folder = BASE / 'Data Sets'
# cleanedds_folder = BASE / 'Data Sets' / 'Cleaned Data'
# networks_folder = BASE / 'Data Sets' / 'Networks'

# Sanity Check

## Sanity Check Test

In [ ]:
# Check how it looks, LINE BY LINE
# CHECKING FOR TYPES OF TWEET THAT ARE NOT ORIGINAL
AItrust_twits_dict_test = open(datasets_folder/'AItrust_twits_dict_test.json','r',encoding='utf-8')

i = 0
for line in AItrust_twits_dict_test:
    twit = json.loads(line)
    #print(twit)
    try:
      i+=1
      print(twit['referenced_tweets'])
      print(twit['type'])
      print(twit.keys())
      print('------')
    except:
      pass
    #print('------')
    if i>5:
        break
AItrust_twits_dict_test.close()

In [ ]:
# Check how it looks, LINE BY LINE
# dict_keys(['description', 'public_metrics', 'created_at', 'id', 'entities', 'name', 'username', 'verified'])
AItrust_author_dict_test = open(datasets_folder/'AItrust_author_dict_test.json','r',encoding='utf-8')

i = 0
for line in AItrust_author_dict_test:
    i+=1
    author = json.loads(line)
    print(author.keys())
    print(author['verified'])
    if i>10:
        break
AItrust_author_dict_test.close()

## Sanity Test Full

In [ ]:
# Check how it looks, LINE BY LINE
# CHECKING FOR TYPES OF TWEET THAT ARE NOT ORIGINAL
AItrust_twits_dict = open(datasets_folder/'AItrust_twits_dict.json','r',encoding='utf-8')

i = 0
for line in AItrust_twits_dict:
    twit = json.loads(line)
    #print(twit)
    try:
      i+=1
      print(twit['referenced_tweets'])
      print(twit['type'])
      print(twit.keys())
      print('------')
    except:
      pass
    #print('------')
    if i>5:
        break
AItrust_twits_dict.close()

In [ ]:
# Check how it looks, LINE BY LINE
# CHECKING FOR TYPES OF TWEET THAT ARE NOT ORIGINAL
AItrust_author_dict = open(datasets_folder/'AItrust_author_dict.json','r',encoding='utf-8')

i = 0
for line in AItrust_author_dict:
    i+=1
    author = json.loads(line)
    print(author.keys())
    print(author['verified'])
    print(author['public_metrics'])
    if i>5:
        break
AItrust_author_dict.close()

# Prunning by Date and Content

I have noticed that there is a lot of rubbish that we want to avoid analysing, and deserves prunning:
- If a tweet about AI refers (retweets, quotes, replies) a tweet before our cutoff date, it will appear in the dataset.
- If a tweet about AI refers a tweet that contains no AI terminology, it will still appear in our dataset.

So I will remove all tweets with no AI language, and before our cutoff date.

## Prunning Functions

In [ ]:
# --- Core keywords (NO bare AI/AGI here) ------------------------------------
# We exclude bare "AI"/"AGI" because they're too permissive ("daily", "aging").
# Instead, we handle them explicitly with _RE_AI_ALLOWED/_RE_AGI_ALLOWED below.
pattern = re.compile(
    r'(?:'                                 # group of alternatives
        r'\bChatGPT\b|'
        r'\bChat-GPT\b|'
        r'\bGPT(?:-?3|-?4)?\b|'            # GPT, GPT3, GPT-3, GPT4, GPT-4
        r'\bLLMs?\b|'                      # LLM, LLMs
        r'\bBARD\b|'
        r'\bBERT\b|'
        r'\bLaMDA\b|'
        r'\bLLaMA\b|'
        r'\bMed-PaLM\b|'
        r'Bing AI|'                        # phrase
        r'artificial intelligence|'        # phrase
        r'large language models'           # phrase
    r')',
    re.IGNORECASE
)

# --- Allowed AI/AGI forms ----------------------------------------------------
# ✅ Allowed (examples):
#   AI / #AI / #AIart / @AI_news / AI-powered
#   AGI / #AGI / #AGI-safety / @AGIteam / AGI-related
#
# ❌ Rejected:
#   "again", "brain", "aging", "engaging" (embedded 'ai'/'agi' inside words)
# Exclude 'ai' preceded by an apostrophe (e.g., j'ai, l'ai)
_RE_AI_ALLOWED  = re.compile(r'(?:(?<![\'])?\bAI\b|#AI[\w-]*|@AI[\w-]*|AI-[A-Za-z])',   re.IGNORECASE)
_RE_AGI_ALLOWED = re.compile(r'(?:\bAGI\b|#AGI[\w-]*|@AGI[\w-]*|AGI-[A-Za-z])', re.IGNORECASE)

# --- Denylist for spammy tokens ----------------------------------------------
# These strings look like allowed forms but are NOT genuine AI/AGI keywords.
# Example: "#Airdrop" matches "#AI…" but should be rejected.
# block "#airdrop", "#airdrops", "#airdrop2025", "@airdrop_io", etc.
_AI_DENYLIST_RE = re.compile(r'(?i)^(?:#|@)airdrop\w*$')


def has_allowed_ai_form(txt: str) -> bool:
    """
    Returns True if the text contains an allowed AI form
    (standalone, hashtag, mention, hyphen), but not if it's in the denylist.
    """
    if _RE_AI_ALLOWED.search(txt):
        for tok in re.findall(r'[@#][A-Za-z0-9_-]+', txt):
            if _AI_DENYLIST_RE.search(tok):
                return False
        return True
    return False

def has_allowed_agi_form(txt: str) -> bool:
    """Same logic as has_allowed_ai_form but for AGI."""
    if _RE_AGI_ALLOWED.search(txt):
        for tok in re.findall(r'[@#][A-Za-z0-9_-]+', txt):
            if _AI_DENYLIST_RE.search(tok):  # in case of overlap
                return False
        return True
    return False

# --- Minimal ES/FR/PT stopword sets (keep short, high-signal) ---------------
STOPWORDS_ES = {
    "gracias","por","para","porque","tambien","también","hola","adios","adiós",
    "buenos","buenas","dias","días","noche","noches","hoy","mañana","ayer",
    "todos","todas","todo","toda","usted","ustedes","nosotros","vosotros",
    "pero","aunque","si","sí","cada","desde","hasta","segun","según","sobre",
    "entre","sin","con","como","cuando","cuándo","donde","dónde","que","qué",
    "del","al","una","uno","unas","unos","este","esta","estas","estos","eso",
}
STOPWORDS_FR = {
    "merci","bonjour","bonsoir","salut","oui","non","pourquoi","parce","que",
    "cest","c'était","c’etait","c'etait","ça","ca","très","tres",
    "je","tu","il","elle","nous","vous","ils","elles",
    "de","du","des","la","le","les","un","une","au","aux","avec","sans","sur","sous",
    "comme","quand","où","ou","pour","entre","depuis","jusqu","chez",
}
STOPWORDS_PT = {
    "ola","olá","oi","obrigado","obrigada","por","para","porque","tambem","também",
    "de","do","da","dos","das","no","na","nos","nas","um","uma","uns","umas",
    "e","ou","mas","como","quando","onde","que","quê","qual","quais",
    "eu","tu","ele","ela","nos","nós","vos","vós","eles","elas","você","vocês",
    "sei","não","nao","sim","muito","pouco","mais","menos","aqui","ali","lá","la",
    "essas","esses","este","esta","isso","aquilo","tudo","nada","sempre","nunca",
}
STOPWORDS_NON_EN = STOPWORDS_ES | STOPWORDS_FR | STOPWORDS_PT

# precompiled token pattern (includes accented letters)
_TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ']+")

# --- Optional langdetect -----------------------------------------------------
def _try_langdetect_is_english(text: str, min_prob: float = 0.85):
    """
    Best-effort language ID using langdetect if available.
    Returns True/False on confident EN/non-EN, or None if unknown/unavailable/low-confidence.
    """
    try:
        import langdetect
        from langdetect import detect_langs
        # Optional: make deterministic
        # langdetect.DetectorFactory.seed = 0
        preds = detect_langs(text)
        if not preds:
            return None
        top = max(preds, key=lambda p: p.prob)
        if top.prob >= min_prob:
            return (top.lang == "en")
        return None
    except Exception:
        return None

def seems_english(
    text: str,
    *,
    # ASCII/diacritics heuristic
    min_ascii_chars: int = 5,
    min_ascii_share: float = 0.75,
    max_accented_share: float = 0.06,      # tighten to 0.04–0.05 to be stricter
    # Tweet-length strategy
    short_text_len: int = 40,              # <40 chars => treat as "short"
    # Stopword thresholds (long texts)
    non_en_stopword_min_hits: int = 2,
    non_en_stopword_min_ratio: float = 0.30,
    # Stopword thresholds (short texts) — stricter to catch brief FR/ES/PT
    short_stopword_min_hits: int = 1,
    short_stopword_min_ratio: float = 0.25,
    # langdetect thresholds for long texts
    use_langdetect: bool = True,
    langdetect_min_prob: float = 0.85,
) -> bool:
    """
    English-likeness heuristic with a split strategy:

    SHORT tweets (len(text) < short_text_len):
      • Skip langdetect (noisy on very short text).
      • Use ASCII/diacritics heuristic.
      • Use FR/ES/PT stopword screen with stricter thresholds
        (short_stopword_min_hits / short_stopword_min_ratio).

    LONGER tweets (len(text) >= short_text_len):
      • Try langdetect first:
          - Confident EN (>= langdetect_min_prob)  -> accept
          - Confident non‑EN                       -> reject
          - Otherwise                              -> fall through
      • Then ASCII/diacritics heuristic.
      • Then FR/ES/PT stopword screen with normal thresholds
        (non_en_stopword_min_hits / non_en_stopword_min_ratio).
    """
    if not text:
        return False

    # Clean text by removing URLs, mentions, and hashtags
    txt_clean = re.sub(r'https?://\S+|[@#]\w+', '', text)

    is_short = len(txt_clean) < short_text_len

    # --- ASCII coverage & diacritics (used in both paths) ---
    ascii_pool = set(string.ascii_letters + string.digits + string.punctuation + " ")
    candidates = [ch for ch in txt_clean if ch.isalnum() or ch in string.punctuation or ch.isspace()]
    if not candidates:
        return False
    ascii_chars = sum(ch in ascii_pool for ch in candidates)
    ascii_ratio = ascii_chars / len(candidates)
    if ascii_chars < min_ascii_chars or ascii_ratio < min_ascii_share:
        return False

    letters = [ch for ch in txt_clean if ch.isalpha()]
    if letters:
        accented_letters = sum(127 < ord(ch) <= 255 for ch in letters)
        accented_ratio = accented_letters / len(letters)
        if accented_ratio > max_accented_share:
            return False

    # --- Branch: short vs long ---
    if is_short:
        # SHORT: rely on stopwords (stricter) after ASCII/diacritics passed
        tokens = [t.lower() for t in _TOKEN_RE.findall(txt_clean)]
        if tokens:
            hits = sum(t in STOPWORDS_NON_EN for t in tokens)
            hit_ratio = hits / len(tokens)
            if hits >= short_stopword_min_hits and hit_ratio >= short_stopword_min_ratio:
                return False
        return True
    else:
        # LONG: try langdetect first (if enabled)
        if use_langdetect:
            is_en = _try_langdetect_is_english(txt_clean, min_prob=langdetect_min_prob)
            if is_en is True:
                return True
            if is_en is False:
                return False
            # None -> fall through

        # Then stopword screen (normal thresholds)
        tokens = [t.lower() for t in _TOKEN_RE.findall(txt_clean)]
        if tokens:
            hits = sum(t in STOPWORDS_NON_EN for t in tokens)
            hit_ratio = hits / len(tokens)
            if hits >= non_en_stopword_min_hits and hit_ratio >= non_en_stopword_min_ratio:
                return False
        return True

def keep_tweet(tw: dict) -> bool:
    """
    Keep a tweet if:
      1) It matches a core keyword in `pattern` OR matches an allowed AI/AGI form (not in denylist); and
      2) It contains at least one art/creative keyword from keywords.txt; and
      3) It's English (lang startswith 'en' OR passes heuristic).

    ✅ Kept examples:
        "I use ChatGPT daily"        -> keyword = ChatGPT
        "AI-powered apps"            -> allowed AI hyphen form
        "#AIart is trending"         -> allowed AI hashtag
        "@AIResearch updates"        -> allowed AI mention
        "AI is the future"           -> standalone AI
        "AGI-safety paper"           -> allowed AGI hyphen form

    ❌ Rejected examples:
        "Brain again daily"          -> 'ai' inside other word, no keyword
        "This trend shows aging..."  -> 'agi' inside other word, no keyword
        "テスト GPT"                   -> non-English (fails heuristic/lang)
        "#Airdrop event"             -> blocked by denylist
        "@Airdrop promo"             -> blocked by denylist
    """
    txt = (tw.get("text") or "").strip()
    if not txt:
        return False

    # 1) Must contain one of the keywords OR an allowed AI/AGI form (with denylist check)
    if not (pattern.search(txt) or has_allowed_ai_form(txt) or has_allowed_agi_form(txt)):
        return False

    # 2) Must contain at least one art/creative keyword from keywords.txt
    if not tweet_has_keyword(txt):
        return False

    # 3) English check
    lang = (tw.get("lang") or "").lower()
    if not (lang.startswith("en") or seems_english(txt)):
        return False

    return True

def keep_tweet_ai_only(tw: dict) -> bool:
    """
    Keep a tweet if:
      1) It matches a core keyword in `pattern` OR matches an allowed AI/AGI form (not in denylist); and
      2) It's English (lang startswith 'en' OR passes heuristic).

    This is the AI-only filter — it does NOT require art/creative keywords.
    Use keep_tweet() if you also want the art/creative keyword requirement.
    """
    txt = (tw.get("text") or "").strip()
    if not txt:
        return False

    # 1) Must contain one of the keywords OR an allowed AI/AGI form (with denylist check)
    if not (pattern.search(txt) or has_allowed_ai_form(txt) or has_allowed_agi_form(txt)):
        return False

    # 2) English check
    lang = (tw.get("lang") or "").lower()
    if not (lang.startswith("en") or seems_english(txt)):
        return False

    return True


## Keywords Filter (from `keywords.txt`)

An **independent, additive filter**: a tweet must contain at least one of the creative/art-related keywords
listed in `keywords.txt`. This is applied in addition to the AI terminology filter above.
Matching is **case-insensitive** and **whole-word** (`\\b` boundaries).


In [ ]:
import re

# --- Keywords filter (from keywords.txt) ------------------------------------
# Independent additive filter: a tweet must contain at least one of these
# creative/art-related keywords. Case-insensitive, whole-word matching.
# 60 unique terms loaded from keywords.txt.

_keywords = [
    "Art",
    "artists",
    "creativity",
    "Anime",
    "Film",
    "Music",
    "Comedy",
    "Satire",
    "Creative",
    "Copyright",
    "Author",
    "Authorship",
    "Aesthetic",
    "Style",
    "Propaganda",
    "Erotic",
    "Image",
    "Images",
    "Photograph",
    "Photography",
    "Print",
    "Portrait",
    "Expression",
    "Soul",
    "Emotion",
    "Gallery",
    "Draw",
    "Paint",
    "Movie",
    "Picture",
    "Character",
    "Artwork",
    "Illustration",
    "Illustrator",
    "Musician",
    "Fiction",
    "NFT",
    "Ugly",
    "Beautiful",
    "Curate",
    "Curator",
    "Curation",
    "Theft",
    "Plagiarism",
    "Imagination",
    "Craft",
    "Meaning",
    "Sculpture",
    "Collage",
    "Performance",
    "Dance",
    "Theatre",
    "Stage",
    "Disney",
    "Pixar",
    "Nintendo",
    "Hayao Miyazaki",
    "Wes Anderson",
    "Studio Ghibli",
    "Hideo Kojima",
]

# Build a single compiled regex: whole-word, case-insensitive
_alts = '|'.join(re.escape(kw) for kw in _keywords)
keywords_pattern = re.compile(r'\b(' + _alts + r')\b', re.IGNORECASE)

def tweet_has_keyword(text: str) -> bool:
    """Return True if the tweet text contains at least one keyword from keywords.txt."""
    return bool(keywords_pattern.search(text))

print(f'Keywords pattern compiled with {len(_keywords)} unique terms.')
print('Example match test ("I love Art and NFTs"):', tweet_has_keyword('I love Art and NFTs'))
print('Example match test ("hello world"):', tweet_has_keyword('hello world'))


In [ ]:
# We only consider tweets on/after 2022-10-31 (offset-aware datetime)
earliest_date = datetime(2022, 10, 31, tzinfo=timezone.utc)
earliest_date = earliest_date.date()

# Preprocessing
def preprocess(text):
    # Convert to lowercase for uniformity
    text = text.lower()
    # Remove the retweet marker 'rt' if it appears at the start of the tweet
    text = re.sub(r'^rt\s+', '', text)
    # Remove URLs entirely (http/https links are noise for text analysis)
    text = re.sub(r'http\S+', '', text)
    # Remove @mentions entirely (including the optional colon that follows them in RTs)
    text = re.sub(r'@\w+:?\s*', '', text)
    # Remove the '#' symbol but keep the hashtag word in the text
    text = re.sub(r'#(\w+)', r'\1', text)
    # Replace newlines with a single space
    text = text.replace('\n', ' ')
    # Collapse any sequence of whitespace into a single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_hashtags(text):
    # Extract all hashtags from the original (unprocessed) text.
    # Returns a list of hashtag words (without the '#' symbol), preserving original casing.
    return re.findall(r'#(\w+)', text)


## Prune Test DS

In [ ]:
# Single pass: write AI-only and AI+Art pruned datasets simultaneously
do_prune = True
if do_prune:
    total_ai = 0
    total_art = 0
    duplicates_skipped = 0
    seen_all_ids = set()
    seen_written = set()

    with open(datasets_folder/'AItrust_twits_dict_test.json','r',encoding='utf-8') as in_f, \
         open(cleanedds_folder/'AItrust_twits_pruned_dict_test.json','w',encoding='utf-8') as ai_f, \
         open(cleanedds_folder/'AItrust_Art_pruned_twit_dict_test.json','w',encoding='utf-8') as art_f:

        for line in tqdm.tqdm(in_f, total=233094):
            try:
                twit = json.loads(line)
                twid = twit.get('id')
                if twid is None:
                    continue

                seen_all_ids.add(twid)

                if twid in seen_written:
                    duplicates_skipped += 1
                    continue

                if keep_tweet_ai_only(twit):
                    tw_date = twit['created_at']
                    date_string = tw_date.replace('Z', '+00:00')
                    date_only = datetime.fromisoformat(date_string).date()
                    if date_only >= earliest_date:
                        twit['processed_text'] = preprocess(twit['text'])
                        ai_f.write(json.dumps(twit, ensure_ascii=False) + '\n')
                        total_ai += 1
                        if tweet_has_keyword(twit['text']):
                            art_f.write(json.dumps(twit, ensure_ascii=False) + '\n')
                            total_art += 1
                        seen_written.add(twid)

            except Exception as e:
                print(e)
                continue

    print(f'\nAI-only written:    {total_ai}')
    print(f'AI+Art written:     {total_art}')
    print(f'\nunique_ids_seen:    {len(seen_all_ids)}')
    print(f'unique_ids_written: {len(seen_written)}')
    print(f'duplicates_skipped: {duplicates_skipped}')


In [ ]:
for label, fname in [
    ('AI-only', 'AItrust_twits_pruned_dict_test.json'),
    ('AI+Art',  'AItrust_Art_pruned_twit_dict_test.json'),
]:
    with open(cleanedds_folder/fname, 'r', encoding='utf-8') as f:
        count = 0
        twids = set()
        for line in tqdm.tqdm(f, desc=label):
            twit = json.loads(line)
            twids.add(twit['id'])
            count += 1
    print(f'{label}: {count} tweets, {len(twids)} unique IDs')


## Prune Full DS

In [ ]:
# Single pass: write AI-only and AI+Art pruned datasets simultaneously
do_prune = True
if do_prune:
    total_ai = 0
    total_art = 0
    duplicates_skipped = 0
    seen_all_ids = set()
    seen_written = set()

    with open(datasets_folder/'AItrust_twits_dict.json','r',encoding='utf-8') as in_f, \
         open(cleanedds_folder/'AItrust_twits_pruned_dict.json','w',encoding='utf-8') as ai_f, \
         open(cleanedds_folder/'AItrust_Art_pruned_twit_dict.json','w',encoding='utf-8') as art_f:

        for line in tqdm.tqdm(in_f, total=36560405):
            try:
                twit = json.loads(line)
                twid = twit.get('id')
                if twid is None:
                    continue

                seen_all_ids.add(twid)

                if twid in seen_written:
                    duplicates_skipped += 1
                    continue

                if keep_tweet_ai_only(twit):
                    tw_date = twit['created_at']
                    date_string = tw_date.replace('Z', '+00:00')
                    date_only = datetime.fromisoformat(date_string).date()
                    if date_only >= earliest_date:
                        twit['processed_text'] = preprocess(twit['text'])
                        ai_f.write(json.dumps(twit, ensure_ascii=False) + '\n')
                        total_ai += 1
                        if tweet_has_keyword(twit['text']):
                            art_f.write(json.dumps(twit, ensure_ascii=False) + '\n')
                            total_art += 1
                        seen_written.add(twid)

            except Exception as e:
                print(e)
                continue

    print(f'\nAI-only written:    {total_ai}')
    print(f'AI+Art written:     {total_art}')
    print(f'\nunique_ids_seen:    {len(seen_all_ids)}')
    print(f'unique_ids_written: {len(seen_written)}')
    print(f'duplicates_skipped: {duplicates_skipped}')


In [ ]:
# Preview a few entries from the pruned AI-only twit dictionary
preview_path = cleanedds_folder / 'AItrust_twits_pruned_dict.json'
n_examples = 13

with open(preview_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= n_examples:
            break
        twit = json.loads(line)
        print(f'--- Entry {i+1} ---')
        for k, v in twit.items():
            print(f'  {k}: {v}')
        print()


# Timelines and Networks

## Test Data

### Generate Test Timeline, Author Corpus and Network Dictionaries

In [ ]:
type_of_network = 'retweeted'

In [ ]:
# Check how it looks, LINE BY LINE
generate_data = True
if generate_data:
  AItrust_pruned_twits_test = open(cleanedds_folder/'AItrust_twits_pruned_dict_test.json','r',encoding='utf-8')
  timeline_dict = dict()
  basic_counts_dict = {'original':0,'retweeted':0,'replied_to':0,'quoted':0,'total_count':0,'exceptions':0}
  network_dict = dict()
  author_corpus_dict = dict()
  tweet_times = []
  twids =set()
  i = 0

  for line in tqdm.tqdm(AItrust_pruned_twits_test):
    try:
        twit = json.loads(line)
        i+=1

        # Ids
        twid=twit['id']
        twids.add(twid)

        # Basic Counts
        type_of_tweet = twit['type']
        basic_counts_dict[type_of_tweet]+=1
        basic_counts_dict['total_count']+=1

        # Timeline
        tw_date = twit['created_at']
        # Removing the 'Z' as fromisoformat doesn't handle the 'Z' which stands for UTC
        date_string = tw_date.replace('Z', '+00:00')
        # Converting the string to a datetime object
        datetime_obj = datetime.fromisoformat(date_string)
        # Convert datetime object to just a date (removes time part)
        date_only = datetime_obj.date()
        #original_tweet_date = datetime.strptime(tweet['TW_Date'],'%a %b %d %X %z %Y').replace(minute=0, second=0, microsecond=0)
        timeline_dict[twid]=tw_date#original_tweet_date
        tweet_times.append(date_only)

        # Network
        if twit['type'] == type_of_network:
          author = str(twit['author_id'])
          #print(author)
          referenced_author=str(twit['referenced_tweets_dictionary']['author_id'])
          #print(referenced)
          if referenced_author in network_dict:
            if author in network_dict[referenced_author]:
                network_dict[referenced_author][author]+=1
            else:
                network_dict[referenced_author][author]=1
          else:
              network_dict[referenced_author]=dict()
              network_dict[referenced_author][author]=1

        # Author Corpus Dict
        author = str(twit['author_id'])
        if author in author_corpus_dict:
          author_corpus_dict[author].append(twit['text'])
        else:
          author_corpus_dict[author]=[]
          author_corpus_dict[author].append(twit['text'])

    except Exception as e:
        #print(traceback.format_exc())
        print(e)
        basic_counts_dict['exceptions']+=1
        continue #goes to the next iteration

  with open(networks_folder / 'test_network_dict.pkl', 'wb') as f:
      pickle.dump(network_dict, f)

  with open(cleanedds_folder / 'test_basic_counts_dict.pkl', 'wb') as f:
      pickle.dump(basic_counts_dict, f)

  with open(cleanedds_folder / 'test_timeline_dict.pkl', 'wb') as f:
      pickle.dump(timeline_dict, f)

  with open(cleanedds_folder / 'test_author_corpus_dict.pkl', 'wb') as f:
    pickle.dump(author_corpus_dict, f)

  AItrust_pruned_twits_test.close()
  print('\n')
  print(len(timeline_dict))
  print('\n')
  print(i)
  print(len(tweet_times))
  print(len(twids))
  print(basic_counts_dict)

### Test Timeline

In [ ]:
with open(cleanedds_folder/'test_timeline_dict.pkl', 'rb') as f:
    timeline_dict = pickle.load(f)

print(len(timeline_dict))
# Select a random key from the dictionary
random_key = random.choice(list(timeline_dict.keys()))
timeline_dict[random_key]

In [ ]:
rdtw = random.sample(list(timeline_dict.items()), 1)
print(rdtw)
example_date = rdtw[0][1]
print(example_date)

# Removing the 'Z' as fromisoformat doesn't handle the 'Z' which stands for UTC
date_string = example_date.replace('Z', '+00:00')

# Converting the string to a datetime object
datetime_obj = datetime.fromisoformat(date_string)
print(datetime_obj)

# Convert datetime object to just a date (removes time part)
date_only = datetime_obj.date()
print(date_only)
# Remove minutes and seconds by setting them to zero
modified_datetime = datetime_obj.replace(hour=0,minute=0, second=0)
print(modified_datetime)

In [ ]:
%%time

tweet_times2 = []
print(len(timeline_dict.keys()))
print('Normalizing to the hour')
for key in tqdm.tqdm(timeline_dict):
    tw_date = timeline_dict[key]
    # Removing the 'Z' as fromisoformat doesn't handle the 'Z' which stands for UTC
    date_string = tw_date.replace('Z', '+00:00')
    # Converting the string to a datetime object
    datetime_obj = datetime.fromisoformat(date_string)
    # Convert datetime object to just a date (removes time part)
    date_only = datetime_obj.date()
    tweet_times2.append(date_only)

In [ ]:
# Create the histogram with a KDE
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.histplot(tweet_times2, kde=True, bins=150, stat="density")
plt.title('Smooth Histogram with KDE')
plt.xlabel('Data Points')
plt.ylabel('Density')
plt.show()

### Test Network

In [ ]:
# Generate Network
%%time

with open(networks_folder/'test_network_dict.pkl', 'rb') as f:
   network_dict = pickle.load(f)

G = nx.DiGraph()
for key in tqdm.tqdm(network_dict):
    G.add_node(key)
    for referenced in network_dict[key]:
        G.add_node(referenced)
        weight=network_dict[key][referenced]
        G.add_edge(referenced,key,weight=weight)

In [ ]:
print(len(G.nodes))
print(len(G.edges))

In [ ]:
%%time
nx.write_gml(G,networks_folder/'Test_Network.gml')
F = nx.read_gml(networks_folder/'Test_Network.gml')
print(G.nodes==F.nodes)
print(G.edges==F.edges)

In [ ]:
%%time
nx.write_graphml(G, networks_folder/"Test_Network.graphml")
F = nx.read_graphml(networks_folder/'Test_Network.graphml')
print(G.nodes==F.nodes)
print(G.edges==F.edges)

In [ ]:
%%time
nx.write_gexf(G, networks_folder/"Test_Network.gexf")
F = nx.read_gexf(networks_folder/'Test_Network.gexf')
print(G.nodes==F.nodes)
print(G.edges==F.edges)

In [ ]:
# https://networkx.org/documentation/networkx-1.10/reference/readwrite.json_graph.html
# https://networkx.org/documentation/networkx-1.10/reference/generated/networkx.readwrite.json_graph.node_link_data.html#networkx.readwrite.json_graph.node_link_data
#https://stackoverflow.com/questions/34665042/read-json-graph-networkx-file
%%time
from networkx.readwrite import json_graph

g_json = json_graph.node_link_data(G)
json.dump(g_json,open(networks_folder/'Test_Network.json','w'))#,indent=2)

with open(networks_folder/'Test_Network.json','r') as f:
    js_graph = json.load(f)

J=json_graph.node_link_graph(js_graph)
print(G.nodes==J.nodes)
print(G.edges==J.edges)

### Test Author Corpus Dictionary

In [ ]:

with open(cleanedds_folder/'test_author_corpus_dict.pkl', 'rb') as f:
   author_corpus_dict = pickle.load(f)

i=0
for key in author_corpus_dict:
  i+=1
  print(author_corpus_dict[key])
  print('-----')
  if i>10:
    break

## Full Data

### Generate Full Timeline and Network Dictionaries

In [ ]:
type_of_network = 'retweeted'

In [ ]:
# Check how it looks, LINE BY LINE
generate_data = True
if generate_data:
  # Open the full dataset file
  with open(cleanedds_folder/'AItrust_twits_pruned_dict.json','r',encoding='utf-8') as AItrust_pruned_twits:
    timeline_dict = dict()
    basic_counts_dict = {'original':0,'retweeted':0,'replied_to':0,'quoted':0,'total_count':0,'exceptions':0}
    network_dict = dict()
    author_corpus_dict = dict()
    tweet_times = []
    twids =set()
    i = 0

    for line in tqdm.tqdm(AItrust_pruned_twits, total=17410035): # total is an ESTIMATE from a previous run (progress-bar only; harmless if stale)
      try:
          twit = json.loads(line)
          i+=1

          # Ids
          twid=twit['id']
          twids.add(twid)

          # Basic Counts
          type_of_tweet = twit['type']
          basic_counts_dict[type_of_tweet]+=1
          basic_counts_dict['total_count']+=1

          # Timeline
          tw_date = twit['created_at']
          # Removing the 'Z' as fromisoformat doesn't handle the 'Z' which stands for UTC
          date_string = tw_date.replace('Z', '+00:00')
          # Converting the string to a datetime object
          datetime_obj = datetime.fromisoformat(date_string)
          # Convert datetime object to just a date (removes time part)
          date_only = datetime_obj.date()
          #original_tweet_date = datetime.strptime(tweet['TW_Date'],'%a %b %d %X %z %Y').replace(minute=0, second=0, microsecond=0)
          timeline_dict[twid]=tw_date#original_tweet_date
          tweet_times.append(date_only)

          # Network
          if twit['type'] == type_of_network:
            author = str(twit['author_id'])
            #print(author)
            referenced_author=str(twit['referenced_tweets_dictionary']['author_id'])
            #print(referenced)
            if referenced_author in network_dict:
              if author in network_dict[referenced_author]:
                  network_dict[referenced_author][author]+=1
              else:
                  network_dict[referenced_author][author]=1
            else:
                network_dict[referenced_author]=dict()
                network_dict[referenced_author][author]=1

          # Author Corpus Dict
          author = str(twit['author_id'])
          if author in author_corpus_dict:
            author_corpus_dict[author].append(twit['text'])
          else:
            author_corpus_dict[author]=[]
            author_corpus_dict[author].append(twit['text'])

      except Exception as e:
          #print(traceback.format_exc())
          print(e)
          basic_counts_dict['exceptions']+=1
          continue #goes to the next iteration

    with open(networks_folder / 'full_network_dict.pkl', 'wb') as f:
        pickle.dump(network_dict, f)

    with open(cleanedds_folder / 'full_basic_counts_dict.pkl', 'wb') as f:
        pickle.dump(basic_counts_dict, f)

    with open(cleanedds_folder / 'full_timeline_dict.pkl', 'wb') as f:
        pickle.dump(timeline_dict, f)

    with open(cleanedds_folder / 'full_author_corpus_dict.pkl', 'wb') as f:
      pickle.dump(author_corpus_dict, f)

    #AItrust_pruned_twits.close() # File is automatically closed by the 'with' statement
    print('\n')
    print(len(timeline_dict))
    print('\n')
    print(i)
    print(len(tweet_times))
    print(len(twids))
    print(basic_counts_dict)

### Full Timeline

In [ ]:
with open(cleanedds_folder/'full_timeline_dict.pkl', 'rb') as f:
    timeline_dict = pickle.load(f)

print(len(timeline_dict))
# Select a random key from the dictionary
random_key = random.choice(list(timeline_dict.keys()))
timeline_dict[random_key]

In [ ]:
rdtw = random.sample(list(timeline_dict.items()), 1)
print(rdtw)
example_date = rdtw[0][1]
print(example_date)

# Removing the 'Z' as fromisoformat doesn't handle the 'Z' which stands for UTC
date_string = example_date.replace('Z', '+00:00')

# Converting the string to a datetime object
datetime_obj = datetime.fromisoformat(date_string)
print(datetime_obj)

# Convert datetime object to just a date (removes time part)
date_only = datetime_obj.date()
print(date_only)
# Remove minutes and seconds by setting them to zero
modified_datetime = datetime_obj.replace(hour=0,minute=0, second=0)
print(modified_datetime)

In [ ]:
%%time
# We only consider tweets after September 2022
# Offset-aware datetime
earliest_date = datetime(2022, 11, 1, tzinfo=timezone.utc)
earliest_date = earliest_date.date()

tweet_times = []
print(len(timeline_dict.keys()))
print('Normalizing to the hour')
for key in tqdm.tqdm(timeline_dict):
    tw_date = timeline_dict[key]
    # Removing the 'Z' as fromisoformat doesn't handle the 'Z' which stands for UTC
    date_string = tw_date.replace('Z', '+00:00')
    # Converting the string to a datetime object
    datetime_obj = datetime.fromisoformat(date_string)
    # Convert datetime object to just a date (removes time part)
    date_only = datetime_obj.date()
    tweet_times.append(date_only)

In [ ]:
%%time
# Create the histogram with a KDE
sns.set(style="whitegrid")
plt.figure(figsize=(10, 8))
sns.histplot(tweet_times, kde=True, bins=150, stat="density")
plt.title('Timeline Smooth Histogram with KDE')
plt.xlabel('Dates')
plt.ylabel('Density')
plt.xticks(fontsize=8,rotation=20)
plt.show()

### Full Network

In [ ]:
# Generate Network
%%time
with open(networks_folder/'full_network_dict.pkl','rb') as f:
   network_dict = pickle.load(f)

G = nx.DiGraph()
for key in tqdm.tqdm(network_dict):
    G.add_node(key)
    for referenced in network_dict[key]:
        G.add_node(referenced)
        weight=network_dict[key][referenced]
        G.add_edge(referenced,key,weight=weight)

In [ ]:
print(len(G.nodes))
print(len(G.edges))

In [ ]:
%%time
nx.write_gml(G,networks_folder/'Full_Network.gml')
F = nx.read_gml(networks_folder/'Full_Network.gml')
print(G.nodes==F.nodes)
print(G.edges==F.edges)

In [ ]:
%%time
nx.write_graphml(G, networks_folder/"Full_Network.graphml")
F = nx.read_graphml(networks_folder/'Full_Network.graphml')
print(G.nodes==F.nodes)
print(G.edges==F.edges)

In [ ]:
# https://networkx.org/documentation/networkx-1.10/reference/readwrite.json_graph.html
# https://networkx.org/documentation/networkx-1.10/reference/generated/networkx.readwrite.json_graph.node_link_data.html#networkx.readwrite.json_graph.node_link_data
#https://stackoverflow.com/questions/34665042/read-json-graph-networkx-file
%%time
from networkx.readwrite import json_graph

g_json = json_graph.node_link_data(G)
json.dump(g_json,open(networks_folder/'Full_Network.json','w'))#,indent=2)

with open(networks_folder/'Full_Network.json','r') as f:
    js_graph = json.load(f)

J=json_graph.node_link_graph(js_graph)
print(G.nodes==J.nodes)
print(G.edges==J.edges)

### Full Author Corpus Dict

In [ ]:

with open(cleanedds_folder/'full_author_corpus_dict.pkl', 'rb') as f:
   author_corpus_dict = pickle.load(f)

i=0
for key in author_corpus_dict:
  i+=1
  print(author_corpus_dict[key])
  print('-----')
  if i>10:
    break

# Disconnect from Runtime

In [ ]:
from datetime import datetime
import pytz
from IPython.display import Javascript

# Get current time in New York
nyc_time = datetime.now(pytz.timezone('America/New_York'))
formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')

# Print and log
print(f"✅ Disconnected from runtime at: {formatted_time}")

# Disconnect Colab runtime
display(Javascript('google.colab.kernel.disconnect()'))